# Pilot SAM2 su video termico ferroviario

Esperimento riproducibile: SAM 2.1 Small, nessun fine-tuning, box prompt al frame iniziale e propagazione su una clip di 20 secondi. Il modello segmenta e traccia un candidato indicato dall'utente; non decide autonomamente cosa sia anomalo.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!test -d sam2 || git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -q -e .

In [ ]:
%cd /content/sam2
!mkdir -p checkpoints
!wget -q --show-progress -O checkpoints/sam2.1_hiera_small.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
!ls -lh checkpoints/sam2.1_hiera_small.pt

Caricare la clip locale `san_donato_pilot_85_105.mp4`, ottenuta dal video originale nell'intervallo 85–105 secondi.

In [ ]:
%cd /content
from google.colab import files
uploaded = files.upload()

In [ ]:
import cv2
import os
import shutil
from PIL import Image
from IPython.display import display

video_path = "/content/san_donato_pilot_85_105.mp4"
frames_dir = "/content/san_donato_frames"

if os.path.exists(frames_dir):
    shutil.rmtree(frames_dir)
os.makedirs(frames_dir)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    cv2.imwrite(os.path.join(frames_dir, f"{frame_count:05d}.jpg"), frame)
    frame_count += 1

cap.release()
print("Frame estratti:", frame_count, "FPS:", fps)
display(Image.open(os.path.join(frames_dir, "00000.jpg")))

In [ ]:
%cd /content/sam2
import torch
from sam2.build_sam import build_sam2_video_predictor

checkpoint = "/content/sam2/checkpoints/sam2.1_hiera_small.pt"
model_config = "configs/sam2.1/sam2.1_hiera_s.yaml"

predictor = build_sam2_video_predictor(model_config, checkpoint, device="cuda")
inference_state = predictor.init_state(
    video_path=frames_dir,
    offload_video_to_cpu=True,
)
predictor.reset_state(inference_state)
print("SAM2 caricato; frame:", len(inference_state["images"]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

first_frame = np.array(Image.open(f"{frames_dir}/00000.jpg").convert("RGB"))
person_box = np.array([392, 195, 438, 270], dtype=np.float32)

plt.figure(figsize=(10, 8))
plt.imshow(first_frame)
x1, y1, x2, y2 = person_box
plt.gca().add_patch(Rectangle((x1, y1), x2-x1, y2-y1, edgecolor="red", facecolor="none", linewidth=2))
plt.title("Box prompt al frame 0")
plt.axis("off")
plt.show()

In [ ]:
predictor.reset_state(inference_state)
with torch.inference_mode():
    _, object_ids, mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=0,
        obj_id=1,
        box=person_box,
    )

mask = (mask_logits[0] > 0).cpu().numpy().squeeze()
overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
overlay[mask] = [0.0, 1.0, 0.0, 0.55]

plt.figure(figsize=(10, 8))
plt.imshow(first_frame)
plt.imshow(overlay)
plt.title("Maschera iniziale SAM2")
plt.axis("off")
plt.show()

In [ ]:
from tqdm.auto import tqdm

video_segments = {}
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    propagation = predictor.propagate_in_video(inference_state)
    for out_frame_idx, out_object_ids, out_mask_logits in tqdm(
        propagation, total=frame_count, desc="Propagazione SAM2"
    ):
        video_segments[out_frame_idx] = {
            int(object_id): (out_mask_logits[index] > 0).cpu().numpy().squeeze()
            for index, object_id in enumerate(out_object_ids)
        }

print("Frame elaborati:", len(video_segments))

In [ ]:
check_frames = [0, 100, 200, 300, 400, 499]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for axis, index in zip(axes.ravel(), check_frames):
    frame = np.array(Image.open(f"{frames_dir}/{index:05d}.jpg").convert("RGB"))
    mask = video_segments[index][1]
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[mask] = [0.0, 1.0, 0.0, 0.55]
    axis.imshow(frame)
    axis.imshow(overlay)
    axis.set_title(f"Frame {index} — {index/fps:.1f} s")
    axis.axis("off")

plt.tight_layout()
plt.savefig("/content/sam2_tracking_six_frames.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
output_video = "/content/sam2_overlay_mp4v.mp4"
first = cv2.imread(f"{frames_dir}/00000.jpg")
height, width = first.shape[:2]
writer = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

for index in tqdm(range(frame_count), desc="Creazione video"):
    frame = cv2.imread(f"{frames_dir}/{index:05d}.jpg")
    mask = video_segments[index][1].astype(bool)
    green = np.array([0, 255, 0], dtype=np.float32)
    frame[mask] = (0.45 * frame[mask].astype(np.float32) + 0.55 * green).astype(np.uint8)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(frame, contours, -1, (0, 255, 0), 2)
    cv2.putText(frame, "SAM2 zero-shot - box prompt al frame 0", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)
    writer.write(frame)

writer.release()
h264_video = "/content/sam2_overlay_h264.mp4"
!ffmpeg -y -loglevel error -i /content/sam2_overlay_mp4v.mp4 -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p -movflags +faststart /content/sam2_overlay_h264.mp4
print("Video:", h264_video)

In [ ]:
import json
import zipfile
import pandas as pd

start_frame_originale = 2125
rows, all_masks = [], []

for clip_frame in range(frame_count):
    mask = video_segments[clip_frame][1].astype(np.uint8)
    all_masks.append(mask)
    y, x = np.where(mask > 0)
    if len(x) == 0:
        continue
    rows.append({
        "frame": start_frame_originale + clip_frame,
        "clip_frame": clip_frame,
        "label": "anomalia",
        "score": 1.0,
        "xtl": int(x.min()), "ytl": int(y.min()),
        "xbr": int(x.max()) + 1, "ybr": int(y.max()) + 1,
        "prompt": "box_person_frame_0",
        "method": "sam2.1_hiera_small",
        "mask_area_pixels": int(mask.sum()),
        "score_note": "valore fisso; non e confidenza di detection",
    })

csv_path = "/content/sam2_boxes.csv"
masks_path = "/content/sam2_masks.npz"
metadata_path = "/content/metadata.json"
bundle_path = "/content/sam2_pilot_results.zip"
pd.DataFrame(rows).to_csv(csv_path, index=False)
np.savez_compressed(masks_path, masks=np.stack(all_masks), source_frames=np.arange(start_frame_originale, start_frame_originale + frame_count))

metadata = {
    "source_video": "test_video_san_donato.mp4",
    "clip_seconds": [85, 105],
    "model": "SAM 2.1",
    "checkpoint": "sam2.1_hiera_small.pt",
    "prompt_type": "box",
    "prompt_box": person_box.tolist(),
    "fine_tuning": False,
    "hardware": "Google Colab Tesla T4",
}
with open(metadata_path, "w") as handle:
    json.dump(metadata, handle, indent=2)

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(h264_video, "sam2_overlay.mp4")
    archive.write(csv_path, "sam2_boxes.csv")
    archive.write(masks_path, "sam2_masks.npz")
    archive.write(metadata_path, "metadata.json")
    archive.write("/content/sam2_tracking_six_frames.png", "sam2_tracking_six_frames.png")

print("Pacchetto:", bundle_path)
files.download(bundle_path)